# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import chromadb

from dotenv import load_dotenv
from pydantic import BaseModel
from typing import Any, Dict, List

from tavily import TavilyClient
import lib.state_machine as sm

from lib.tooling import tool

In [3]:
load_dotenv(".env") or load_dotenv("config.env")

TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
if not TAVILY_API_KEY:
    raise EnvironmentError("❌ TAVILY_API_KEY not found in environment variables")

print("✅ Environment variables loaded successfully (Tavily OK)")

✅ Environment variables loaded successfully (Tavily OK)


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [4]:
CHROMA_PATH = "chroma_udaplay_db_local"
COLLECTION_NAME = "udaplay"

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_collection(name=COLLECTION_NAME)

@tool
def retrieve_game(query: str, k: int = 5) -> list:
    """
    Semantic search: Finds most results in the vector DB
    args:
    - query: a question about game industry.
    """
    results = collection.query(
        query_texts=[query],
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )

    docs = results.get("documents", [[]])[0]
    metas = results.get("metadatas", [[]])[0]
    dists = results.get("distances", [[]])[0]

    out = []
    for doc, meta, dist in zip(docs, metas, dists):
        out.append({
            "Platform": meta.get("Platform"),
            "Name": meta.get("Name"),
            "YearOfRelease": meta.get("YearOfRelease"),
            "Description": meta.get("Description", doc),
            "distance": float(dist),
            "source": "vectordb"
        })
    return out

print("✅ Chroma ready:", CHROMA_PATH, "| collection:", COLLECTION_NAME)


✅ Chroma ready: chroma_udaplay_db_local | collection: udaplay


#### Evaluate Retrieval Tool

In [5]:
 class EvaluationReport(BaseModel):
    useful: bool
    description: str

In [6]:
@tool
def evaluate_retrieval(question: str, retrieved_docs: list) -> dict:
    if not retrieved_docs:
        return EvaluationReport(
            useful=False,
            description="No documents retrieved from VectorDB. Use web search."
        ).model_dump()

    distances = [d.get("distance") for d in retrieved_docs if isinstance(d.get("distance"), (int, float))]
    if not distances:
        return EvaluationReport(
            useful=False,
            description="No distance info in retrieved docs. Use web search."
        ).model_dump()

    best = min(distances)
    if best <= 0.35:
        return EvaluationReport(
            useful=True,
            description=f"Good retrieval (best distance={best:.3f}). Use VectorDB."
        ).model_dump()

    return EvaluationReport(
        useful=False,
        description=f"Weak retrieval (best distance={best:.3f}). Use web search."
    ).model_dump()

#### Game Web Search Tool

In [7]:


tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

@tool
def game_web_search(question: str, max_results: int = 5, search_depth: str = "basic") -> list:
    response = tavily_client.search(
        query=question,
        max_results=max_results,
        search_depth=search_depth,
        include_answer=False,
        include_raw_content=False,
        include_images=False
    )

    results = []
    for r in response.get("results", []):
        results.append({
            "title": r.get("title"),
            "url": r.get("url"),
            "content": r.get("content"),
            "score": r.get("score"),
            "source": "tavily"
        })

    return results

In [8]:
game_web_search("When were Pokémon Gold and Silver released?", max_results=3)


[{'title': 'Pokémon Gold and Silver released October 15, 2000 - 21 years ago ...',
  'url': 'https://www.reddit.com/r/Gameboy/comments/q8rmer/pok%C3%A9mon_gold_and_silver_released_october_15_2000/',
  'content': 'Pokémon Gold and Silver released October 15, 2000 - 21 years ago these games came out and I still love them.',
  'score': 0.9998104,
  'source': 'tavily'},
 {'title': 'Today Makes 26 years since Pokémon Gold & Silver were first release',
  'url': 'https://www.facebook.com/PokemonGlobalNews/posts/today-makes-26-years-since-pok%C3%A9mon-gold-silver-were-first-release/852319867446491/',
  'content': 'November 21, 1999. 25 years ago, Pokémon Gold and Silver versions were released in Japan. · Pokemon Red & Blue were both released on this day in',
  'score': 0.99958915,
  'source': 'tavily'},
 {'title': 'Pokémon Gold and Silver - Wikipedia',
  'url': 'https://en.wikipedia.org/wiki/Pok%C3%A9mon_Gold_and_Silver',
  'content': '***Pokémon Gold Version*** and ***Pokémon Silver Version**

In [9]:
def step_retrieve(state: dict) -> dict:
    """
    Retrieve information from VectorDB using the current question.
    If the question is a follow-up, enrich it with the last game discussed.
    """
    question = state["question"]
    history = state.get("history", [])
    last_game = state.get("last_game", "")

    # Detect follow-up questions that need conversational context
    follow_up_keywords = [
        "plataforma", "lanzó", "lanzamiento", "originalmente",
        "platform", "released", "release", "year", "publisher", "editor"
    ]

    is_follow_up = any(word in question.lower() for word in follow_up_keywords)

    # ✅ Use only the last known game, not the full history
    if is_follow_up and last_game:
        full_query = f"{last_game} original release platform {question}"
    elif is_follow_up and history:
        # fallback if last_game is not available
        last_question = history[-1].get("question", "") if isinstance(history[-1], dict) else str(history[-1])
        full_query = f"{last_question} {question}"
    else:
        full_query = question

    retrieved_docs = retrieve_game(full_query)

    # Store the best candidate game from retrieval
    candidate_game = ""
    if retrieved_docs:
        candidate_game = retrieved_docs[0].get("Name", "")

    return {
        **state,
        "retrieved_docs": retrieved_docs,
        "context_query": full_query,
        "candidate_game": candidate_game
    }


def step_evaluate(state: dict) -> dict:
    question = state["question"]
    retrieved_docs = state.get("retrieved_docs", [])
    evaluation = evaluate_retrieval(question, retrieved_docs)
    return {**state, "evaluation": evaluation}


def step_route(state: dict) -> dict:
    evaluation = state.get("evaluation", {})
    route = "vectordb" if evaluation.get("useful") else "web"
    return {**state, "route": route}


def step_web_search(state: dict) -> dict:
    """
    Web fallback also uses the context-enriched query when available.
    """
    query = state.get("context_query") or state["question"]
    web_results = game_web_search(query)
    return {**state, "web_results": web_results}


def step_finalize(state: dict) -> dict:
    """
    Produce final answer and update conversation memory.
    """
    history = state.get("history", [])
    question = state["question"]
    route = state.get("route")

    last_game = state.get("last_game", "")
    candidate_game = state.get("candidate_game", "")

    if route == "vectordb":
        docs = state.get("retrieved_docs", [])
        if not docs:
            history.append({
                "question": question,
                "route": route,
                "game": last_game,
                "context_query": state.get("context_query", question)
            })

            return {
                **state,
                "history": history,
                "source": "vectordb",
                "answer": "No VectorDB docs found.",
                "last_game": last_game
            }

        top = docs[0]
        game_name = top.get("Name", candidate_game or last_game)

        answer = (
            f"Answer based on internal knowledge (VectorDB):\n"
            f"- Game: {top.get('Name')}\n"
            f"- Platform: {top.get('Platform')}\n"
            f"- Year of Release: {top.get('YearOfRelease')}\n"
            f"- Description: {top.get('Description')}"
        )

        history.append({
            "question": question,
            "route": route,
            "game": game_name,
            "context_query": state.get("context_query", question)
        })

        return {
            **state,
            "history": history,
            "source": "vectordb",
            "answer": answer,
            "last_game": game_name
        }

    # Web fallback
    results = state.get("web_results", [])
    game_name = candidate_game or last_game

    if not results:
        history.append({
            "question": question,
            "route": route,
            "game": game_name,
            "context_query": state.get("context_query", question)
        })

        return {
            **state,
            "history": history,
            "source": "web",
            "answer": "No reliable information was found on the web.",
            "last_game": game_name
        }

    top = results[0]
    answer = (
        f"Answer based on web search:\n"
        f"- Title: {top.get('title')}\n"
        f"- Summary: {top.get('content')}\n"
        f"- URL: {top.get('url')}"
    )

    history.append({
        "question": question,
        "route": route,
        "game": game_name,
        "context_query": state.get("context_query", question),
        "citation": top.get("url")
    })

    return {
        **state,
        "history": history,
        "source": "web",
        "answer": answer,
        "last_game": game_name
    }

In [10]:
class UdaPlayState:
    question: str
    history: list
    retrieved_docs: List[Dict[str, Any]]
    evaluation: Dict[str, Any]
    route: str
    web_results: List[Dict[str, Any]]
    answer: str
    source: str

    
# ✅ Conversation context fields
    last_game: str
    candidate_game: str
    context_query: str


udaplay_agent = sm.StateMachine(state_schema=UdaPlayState)

entry = sm.EntryPoint()

retrieve_step = sm.Step("retrieve", step_retrieve)
evaluate_step = sm.Step("evaluate", step_evaluate)
route_step = sm.Step("route", step_route)
web_step = sm.Step("web_search", step_web_search)
final_step = sm.Step("finalize", step_finalize)
end = sm.Termination()

# ✅ Registro manual robusto: usa step_id real (evita KeyError '__termination__')
udaplay_agent.steps[entry.step_id] = entry

for s in [retrieve_step, evaluate_step, route_step, web_step, final_step]:
    udaplay_agent.steps[s.step_id] = s

udaplay_agent.steps[end.step_id] = end

# Entry → Retrieve
udaplay_agent.connect(entry, retrieve_step, condition=lambda state: ["retrieve"])

# Flow
udaplay_agent.connect(retrieve_step, evaluate_step)
udaplay_agent.connect(evaluate_step, route_step)

# Route
udaplay_agent.connect(
    route_step, final_step,
    condition=lambda state: ["finalize"] if state.get("route") == "vectordb" else []
)
udaplay_agent.connect(
    route_step, web_step,
    condition=lambda state: ["web_search"] if state.get("route") == "web" else []
)

udaplay_agent.connect(web_step, final_step)
udaplay_agent.connect(final_step, end)

print("✅ UdaPlay Agent built successfully")

✅ UdaPlay Agent built successfully


### Agent

In [11]:
# ✅ Stateful invoke: accepts a full state dict and returns the final state dict
def udaplay_invoke(state: dict) -> dict:
    run = udaplay_agent.run(state)
    return run.get_final_state()

In [12]:
# ✅ Helper to run the agent and print a structured reasoning report
def run_and_report(question: str):
    state = {
        "question": question,
        "history": [],
        "retrieved_docs": [],
        "evaluation": {},
        "route": "",
        "web_results": [],
        "answer": "",
        "source": "",
        "last_game": "",
        "candidate_game": "",
        "context_query": ""
    }

    final_state = udaplay_invoke(state)

    route = final_state.get("route", "")
    source = final_state.get("source", "")
    retrieved = final_state.get("retrieved_docs", [])
    evaluation = final_state.get("evaluation", {})
    web_results = final_state.get("web_results", [])
    answer = final_state.get("answer", "")
    context_query = final_state.get("context_query", question)

    print("\n" + "=" * 100)
    print("🧠 Pregunta:")
    print(question)

    print("\n🔎 Consulta usada por el agente:")
    print(context_query)

    print("\n📥 Paso 1: Recuperación con VectorDB")
    if retrieved:
        for i, doc in enumerate(retrieved[:3], start=1):
            name = doc.get("Name", "Unknown")
            platform = doc.get("Platform", "Unknown")
            year = doc.get("YearOfRelease", "Unknown")
            distance = doc.get("distance", None)

            distance_text = f"{distance:.3f}" if isinstance(distance, (int, float)) else "N/A"

            print(f"  {i}. {name} | {platform} | {year} | distance={distance_text}")
    else:
        print("  No se recuperaron documentos desde VectorDB.")

    print("\n🧪 Paso 2: Evaluación de la recuperación")
    print(evaluation)

    print("\n🔀 Paso 3: Decisión de ruta")
    print("Ruta seleccionada →", route)

    if route == "web":
        print("\n🌐 Paso 4: Fallback web")
        if web_results:
            top_web = web_results[0]
            print("Fuente web:", top_web.get("url"))
            print("Título:", top_web.get("title"))
        else:
            print("No hubo resultados web.")
    else:
        print("\n🌐 Paso 4: Fallback web")
        print("No usado. La respuesta se generó desde VectorDB.")

    print("\n✅ Paso 5: Respuesta final")
    print(answer)

    print("\n📌 Fuente final")
    if route == "web" and web_results:
        print("Web →", web_results[0].get("url"))
    elif route == "vectordb" and retrieved:
        top = retrieved[0]
        print(
            "VectorDB → "
            f"{top.get('Name')} | {top.get('Platform')} | {top.get('YearOfRelease')}"
        )
    else:
        print(source or "Fuente no disponible")

## ✅ Conversational State Management

The agent maintains a shared `state` object across multiple turns.
Each execution updates the conversation history and reuses it in subsequent
questions.

In the example above:
- The first question initializes the conversation.
- The second question reuses the same state and depends on the context
  established by the first interaction.

This demonstrates long-term conversational state handling using a
State Machine–based agent.


In [13]:
# ✅ Remove any previous version of run_and_report from memory
try:
    del run_and_report
    print("✅ Previous run_and_report deleted")
except NameError:
    print("ℹ️ No previous run_and_report found")

✅ Previous run_and_report deleted


In [14]:
# ✅ Helper to run the agent and print a structured reasoning report
# ✅ REPORT VERSION: V4 - forced route-safe source handling

RUN_AND_REPORT_VERSION = "V4 - forced route-safe source handling"

def run_and_report(question: str):
    state = {
        "question": question,
        "history": [],
        "retrieved_docs": [],
        "evaluation": {},
        "route": "",
        "web_results": [],
        "answer": "",
        "source": "",
        "last_game": "",
        "candidate_game": "",
        "context_query": ""
    }

    final_state = udaplay_invoke(state)

    route = final_state.get("route", "")
    route_norm = str(route).strip().lower()

    retrieved = final_state.get("retrieved_docs", [])
    evaluation = final_state.get("evaluation", {})
    web_results = final_state.get("web_results", [])
    answer = final_state.get("answer", "")
    context_query = final_state.get("context_query", question)

    print("\n" + "=" * 90)
    print(f"✅ REPORT VERSION: {RUN_AND_REPORT_VERSION}")

    print("\n🧠 Pregunta:")
    print(question)

    print("\n🔎 Consulta usada por el agente:")
    print(context_query)

    print("\n📥 Paso 1: Recuperación con VectorDB")
    if retrieved:
        for i, doc in enumerate(retrieved[:3], start=1):
            distance = doc.get("distance")
            distance_text = f"{distance:.3f}" if isinstance(distance, (int, float)) else "N/A"
            print(
                f"{i}. {doc.get('Name')} | "
                f"{doc.get('Platform')} | "
                f"{doc.get('YearOfRelease')} | "
                f"distance={distance_text}"
            )
    else:
        print("No se recuperaron documentos desde VectorDB.")

    print("\n🧪 Paso 2: Evaluación")
    print(evaluation)

    print("\n🔀 Paso 3: Decisión")
    print("Ruta seleccionada:", route_norm)

    print("\n🌐 Paso 4: Uso de herramienta web")
    if route_norm == "web":
        print("Fallback web usado ✅")
        if web_results:
            print("Fuente web:", web_results[0].get("url"))
        else:
            print("Sin resultados web.")
    elif route_norm == "vectordb":
        print("Fallback web NO usado. Se responde desde VectorDB.")
    else:
        print("Ruta no reconocida.")

    print("\n✅ Paso 5: Respuesta final")
    # Print only a compact answer to avoid output mixing/truncation
    answer_lines = str(answer).splitlines()
    for line in answer_lines[:6]:
        print(line)

    print("\n📌 Fuente final")

    # ✅ Forced route-safe source logic
    if route_norm == "vectordb":
        if retrieved:
            top = retrieved[0]
            print(
                "VectorDB → "
                f"{top.get('Name')} | "
                f"{top.get('Platform')} | "
                f"{top.get('YearOfRelease')}"
            )
        else:
            print("VectorDB → sin documentos recuperados")

    elif route_norm == "web":
        if web_results:
            print("Web →", web_results[0].get("url"))
        else:
            print("Web → sin resultados")

    else:
        print("Fuente no disponible")

    print("=" * 90)

### (Optional) Advanced

In [15]:
# ✅ Demo: Agent performance and reasoning on 3 example queries

print("=" * 100)
print("🎮 UdaPlay Agent Demo: Performance and Reasoning Report")
print("=" * 100)

example_questions = [
    "When Pokémon Gold and Silver was released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?"
]

for q in example_questions:
    run_and_report(q)

🎮 UdaPlay Agent Demo: Performance and Reasoning Report
[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: evaluate
[StateMachine] Executing step: route
[StateMachine] Executing step: finalize
[StateMachine] Terminating: __termination__

✅ REPORT VERSION: V4 - forced route-safe source handling

🧠 Pregunta:
When Pokémon Gold and Silver was released?

🔎 Consulta usada por el agente:
When Pokémon Gold and Silver was released?

📥 Paso 1: Recuperación con VectorDB
1. Pokémon Gold and Silver | Game Boy Color | 1999 | distance=0.304
2. Pokémon Ruby and Sapphire | Game Boy Advance | 2002 | distance=0.564
3. Super Mario 64 | Nintendo 64 | 1996 | distance=0.751

🧪 Paso 2: Evaluación
{'useful': True, 'description': 'Good retrieval (best distance=0.304). Use VectorDB.'}

🔀 Paso 3: Decisión
Ruta seleccionada: vectordb

🌐 Paso 4: Uso de herramienta web
Fallback web NO usado. Se responde desde VectorDB.

✅ Paso 5: Respuesta final
Answer based on i

## ✅ Agent Performance and Reasoning Report

The report above demonstrates the agent execution on multiple example queries in a structured and reviewable way.

For each query, the notebook shows the complete reasoning flow:

1. **Original user question**  
   The initial question provided to the agent.

2. **Query used by the agent**  
   The query actually used during retrieval, including conversational context when applicable.

3. **VectorDB retrieval results**  
   The semantic search results returned from the Vector Database, including the most relevant games and their similarity distance.

4. **Retrieval evaluation**  
   The evaluation step that determines whether the retrieved VectorDB results are useful enough to answer the question.

5. **Routing decision**  
   The decision made by the agent to either answer from VectorDB or use the web fallback.

6. **Web fallback usage**  
   Whether the Tavily web search tool was used, including the web source when applicable.

7. **Final answer and source**  
   The final structured answer produced by the agent, along with the source used to support it.

This demonstrates that the agent's reasoning, tool usage, routing logic, and final response are transparent, explainable, and easy to review.

## Long-Term Memory (Future Improvement)

This agent currently uses short-term state management within each execution
of the StateMachine.

As a future improvement, long-term memory could be implemented by:
- Persisting previous answers to disk (JSON / DB)
- Storing conversation summaries in a Vector Database
- Reusing past answers during new queries

Long-term memory is not required for the current project scope.
